# Agentic Metric Evaluation — demo

The agent loop now lives in the `eval` package (see `eval/README.md`). This
notebook is a thin demo: it imports the package, runs the agent on one output,
and shows the DuckDB summary over the Parquet run-rows. The original
`MetricEval.ipynb` is kept as scratch/history.

**Before running:** start the MCP server (`./run_metric_mcp.sh`) and a model
(`sbatch ../run_nim_server.slurm` for the `nim` backend), then paste the MCP URL
below.

In [ ]:
import sys
sys.path.insert(0, "..")  # repo root, so `import analysis...` resolves

from dotenv import load_dotenv
import weave

from eval import config
from eval.runner import aprepare, discover_rows, run_batch

load_dotenv()

In [ ]:
BACKEND = "nim"  # or "playground"
MCP_URL = "http://127.0.0.1:PORT/mcp"  # paste from ./run_metric_mcp.sh

weave.init(config.WEAVE_PROJECT)
ctx = await aprepare(BACKEND, MCP_URL)
print(f"model={ctx['model']}  tools_hash={ctx['tools_hash']}  git={ctx['git_commit']}")

## Run the agent on one output

`discover_rows` lists work via the `list_outputs` MCP tool; `run_batch` runs the
agent, applies the integrity checks + selection scorer, traces to Weave, and
writes one Parquet row per run.

In [ ]:
rows = await discover_rows(MCP_URL, ["5"], limit=1)
results = await run_batch(rows, ctx, concurrency=1, verbose=True)
results[0]["answer"]

## Analytics: DuckDB over the Parquet run-rows

The aggregate (OLAP) view across every run written so far. For a parallel batch,
raise `concurrency`; for the sequential-vs-parallel experiment, run
`python -m analysis.sweep ...` then re-run `concurrency_summary`.

In [ ]:
from analysis.queries import connect, summary_by, concurrency_summary

con = connect()
display(summary_by(con, "backend", "reasoning_level"))
display(concurrency_summary(con))